# Stage 1c — cross-validated reduced-rank KV prediction

This CPU-only notebook reuses the completed Stage 1b sufficient statistics. It fits position-conditioned, reduced-rank student-to-teacher KV maps on one split and evaluates them on the untouched split in both directions. It does not load a model, use a GPU, or update weights.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/0x0shephard/latent-reasoning.git'
RUN_COMMIT = 'main'  # Replace with the printed SHA for a reproducible run.
REPO_DIR = '/content/latent-reasoning'
DRIVE_ROOT = '/content/drive/MyDrive/CODI_KAVA'
STAGE1B_STATISTICS = f'{DRIVE_ROOT}/outputs/stage1b_kv_cross_subspaces'
REPORT_JSON = f'{DRIVE_ROOT}/reports/stage1c_kv_reduced_rank.json'


In [ ]:
import os, pathlib, subprocess, sys

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', RUN_COMMIT], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', os.path.join(REPO_DIR, 'requirements.txt')
], check=True)
os.chdir(REPO_DIR)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Checked out:', commit)
if RUN_COMMIT == 'main':
    print('Pin RUN_COMMIT before recording the final result:', commit)


In [ ]:
from pathlib import Path
import torch

statistics_path = Path(STAGE1B_STATISTICS)
if statistics_path.is_dir():
    statistics_path = statistics_path / 'statistics.pt'
assert statistics_path.is_file(), f'Missing Stage 1b statistics: {statistics_path}'
assert Path('scripts/analyze_kv_reduced_rank.py').is_file(), 'Push Stage 1c first'
payload = torch.load(statistics_path, map_location='cpu', weights_only=False)
assert payload.get('complete') is True, 'Stage 1b extraction is incomplete'
assert payload.get('processed_examples') == 5000, payload.get('processed_examples')
print('Statistics:', statistics_path)
print('Processed examples:', payload['processed_examples'])
print('Runtime:', 'CPU')


In [ ]:
import datetime, time

logs = pathlib.Path(DRIVE_ROOT) / 'logs' / 'stage1c_kv_reduced_rank'
logs.mkdir(parents=True, exist_ok=True)
log_path = logs / 'analyze_n5000.log'
cmd = [
    sys.executable, 'scripts/analyze_kv_reduced_rank.py',
    '--statistics', STAGE1B_STATISTICS,
    '--output', REPORT_JSON,
]
print('Starting:', ' '.join(cmd), flush=True)
print('Persistent log:', log_path, flush=True)
started = time.monotonic()
with log_path.open('a', encoding='utf-8', buffering=1) as log:
    log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(cmd)} ===\n")
    process = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
    return_code = process.wait()
assert return_code == 0, f'Analysis failed with exit code {return_code}'
print(f'Elapsed: {time.monotonic() - started:.1f}s')


In [ ]:
import json
from IPython.display import Markdown, display

report_path = Path(REPORT_JSON)
markdown_path = report_path.with_suffix('.md')
assert report_path.is_file() and markdown_path.is_file()
report = json.loads(report_path.read_text())
display(Markdown(markdown_path.read_text()))
print('\nGATE')
print(json.dumps(report['gate'], indent=2))
print('\nCOMPARISONS')
print(json.dumps(report['comparisons'], indent=2))
